# 08 — SHAP interpretation of the underlying forests

Explain and average the uncalibrated Random Forest components of E2-GP.
These attributions do not exactly decompose the final calibrated probabilities.
A nonlinear monotonic transformation does not guarantee preservation of SHAP
values or feature-importance rankings. The calculations below can be expensive
even when light-curve features and AL results are already cached.


## 0. Configuration


In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'config.py').exists() and (p/'src').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import config
from src.models import classification
from src.models.classification import CLASS_NAMES
from src.interpretability import shap_utils as su
su.thesis_plot_style()            
RS = config.RANDOM_STATE
FIG = config.FIGURE_DIR


## 1. Operational model and evaluation data


In [ ]:
tr = classification.features_e2gp('train')
X_tr, COLS = classification.e2gp_matrix(tr); y_tr = tr['target'].to_numpy()
RETRAIN_MODEL = False
if RETRAIN_MODEL:
    clf = classification.e2gp_al(y=y_tr, random_state=RS).fit(X_tr, y_tr)
else:
    clf, saved_columns, model_contract = classification.load_e2gp_artifact()
    if list(COLS) != list(saved_columns):
        raise ValueError('Current features do not match the saved model contract')
    print('Loaded the verified canonical model; classifier fitting was not repeated.')
NOMBRES_CLASES = [CLASS_NAMES.get(cl, str(cl)) for cl in clf.classes_]

rutas_b4 = config.BatchConfig(4)
te = classification.features_e2gp('test', batch=4)
meta_te = pd.read_csv(rutas_b4.test_metadata_clean_csv)
te = te.merge(meta_te[['object_id', 'true_target']], on='object_id')
X_te, _ = classification.e2gp_matrix(te, COLS)
tt = te['true_target'].to_numpy()
es_anom = tt >= 990
print(f'train {X_tr.shape} - {len(COLS)} features - {len(clf.classes_)} classes')
print(f'test  {X_te.shape} - anomalies (class 99): {int(es_anom.sum())} - known: {int((~es_anom).sum())}')


## 2. Global feature importance


In [ ]:
expl_tr, _ = su.explain_model(clf, X_tr, COLS, NOMBRES_CLASES,
                                max_samples=300, random_state=RS)
imp = su.global_importance(expl_tr, top=20, color_by_family=True,
                            save=FIG / '08m_importancia.png')
plt.show()
imp.head(15)


## 3. A selected feature–class dependence plot


In [ ]:
su.class_dependence(expl_tr, clf, 'baseline_days', class_codes=[88, 90],
                    class_names_list=NOMBRES_CLASES,
                    save=FIG / '08m_dependence_baseline.png')
plt.show()


## 3b. Systematic feature directions by class


In [ ]:
dir_tabla = su.direction_table(expl_tr, clf, NOMBRES_CLASES, top=8)
dir_tabla.to_csv(config.DIR_PROCESSED / '08_direccion_features.csv', index=False)

sort_order = ['AGN','Mira','RRL','EB','M-dwarf','SLSN-I','SNIa','SNII','SNIbc',
         'SNIax','SNIa-91bg','KN','TDE','uLens']   
su.direction_heatmap(expl_tr, clf, NOMBRES_CLASES, top=14, class_order=sort_order,
                       separator_index=5, save=FIG / '08m_heatmap_direcciones.png')
plt.show()
dir_tabla.head(16)


## 4. Class-support diagnostics for known objects and anomalies

Change the sign of the forest attributions for the class selected by the
calibrated model. Compare those descriptive values across known and 99x objects.
They do not exactly explain `1 - max(p_calibrated)` or a changing maximum class.


In [ ]:
rng = np.random.default_rng(RS)
idx_conoc = rng.choice(np.where(~es_anom)[0],
                       size=min(300, int((~es_anom).sum())), replace=False)
sel = np.sort(np.concatenate([np.where(es_anom)[0], idx_conoc]))

expl_te, _ = su.explain_model(clf, X_te[sel], COLS, NOMBRES_CLASES)
tabla_nov = su.class_support_profile(expl_te, clf, X_te[sel], es_anom[sel], top=15,
                              plot_style='dumbbell', save=FIG / '08m_novedad_dumbbell.png')
plt.show()
tabla_nov.head(15)


## 5. Individual objects: side-by-side contributions


In [ ]:
conf = clf.predict_proba(X_te[sel]).max(axis=1)
loc_anom  = np.where(es_anom[sel])[0]
loc_conoc = np.where(~es_anom[sel])[0]
p_anom = int(loc_anom[np.argmax(1 - conf[loc_anom])])   
p_tip  = int(loc_conoc[np.argmax(conf[loc_conoc])])     

plot_titles = [f'Typical object  -  confidence = {conf[p_tip]:.2f}',
           f'Anomaly (class 99)  -  novelty = {1 - conf[p_anom]:.2f}']
su.compare_waterfalls(expl_te, clf, X_te[sel], [p_tip, p_anom],
                       plot_titles=plot_titles, top=10,
                       save=FIG / '08m_waterfalls.png')
plt.show()


## Interpretation limits

The plots describe discriminative components and associations, not physical
causation. Reading anomalies as a lack of known-class support is interpretative.
An exact explanation of the operational novelty function would require a
different analysis of that function of the complete calibrated model.
